In [ ]:
# 基础库导入
import json
import random
import tarfile
import urllib.request
from datetime import datetime
from pathlib import Path

# 数据处理、可视化和 PyTorch
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image, ImageOps
import torch
from torch import nn
from torch.nn import functional as F
from torch.optim import SGD
from torch.utils.data import DataLoader, Dataset
import torchvision.transforms.functional as TF
from torchvision.models import ResNet50_Weights
from torchvision.models.segmentation import DeepLabV3_ResNet50_Weights, deeplabv3_resnet50

# 根据当前工作目录自动定位 exp2 目录
PROJECT_ROOT = Path.cwd()
EXP_ROOT = PROJECT_ROOT / "exp2" if (PROJECT_ROOT / "exp2").exists() else PROJECT_ROOT
print("EXP_ROOT:", EXP_ROOT)


In [ ]:
# 数据准备配置
DOWNLOAD_CONFIG = {
    "data_root": EXP_ROOT / "VOC2012",
    "downloads_dir": EXP_ROOT / "downloads",
    "url": "https://host.robots.ox.ac.uk/pascal/VOC/voc2012/VOCtrainval_11-May-2012.tar",
}

# 实验公共配置
EXPERIMENT_CONFIG = {
    "data_root": EXP_ROOT / "VOC2012",
    "output_root": EXP_ROOT / "outputs",
    "experiment_name": None,
    "seed": 42,
    "device": "cuda" if torch.cuda.is_available() else "cpu",
}

# 训练配置
TRAIN_CONFIG = {
    "train_split": "train",
    "val_split": "val",
    "epochs": 3,
    "batch_size": 4,
    "num_workers": 2,
    "crop_size": 513,
    "min_scale": 0.5,
    "max_scale": 2.0,
    "lr": 0.001,
    "num_classes": 21,
    "ignore_label": 255,
    "weights": "voc",
    "backbone_weights": "imagenet",
}

# 评估配置
EVALUATE_CONFIG = {
    "split": "val",
}

# 推理配置
PREDICT_CONFIG = {
    "input": EXP_ROOT / "VOC2012" / "JPEGImages",
    "long_size": 513,
    "max_images": 5,
}


In [ ]:
# VOC 调色板和类别名
VOC_COLORMAP = [
    (0, 0, 0), (128, 0, 0), (0, 128, 0), (128, 128, 0), (0, 0, 128),
    (128, 0, 128), (0, 128, 128), (128, 128, 128), (64, 0, 0), (192, 0, 0),
    (64, 128, 0), (192, 128, 0), (64, 0, 128), (192, 0, 128), (64, 128, 128),
    (192, 128, 128), (0, 64, 0), (128, 64, 0), (0, 192, 0), (128, 192, 0),
    (0, 64, 128),
]
VOC_CLASSES = [
    "background", "aeroplane", "bicycle", "bird", "boat", "bottle", "bus", "car", "cat",
    "chair", "cow", "diningtable", "dog", "horse", "motorbike", "person", "pottedplant",
    "sheep", "sofa", "train", "tvmonitor",
]
IMAGE_MEAN = (0.485, 0.456, 0.406)
IMAGE_STD = (0.229, 0.224, 0.225)


def ensure_dir(path):
    path = Path(path)
    path.mkdir(parents=True, exist_ok=True)
    return path


def build_experiment_dir():
    name = EXPERIMENT_CONFIG["experiment_name"] or datetime.now().strftime("%Y%m%d_%H%M%S")
    return ensure_dir(EXPERIMENT_CONFIG["output_root"] / name)


def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def save_json(data, path):
    path = Path(path)
    ensure_dir(path.parent)
    with path.open("w", encoding="utf-8") as f:
        json.dump(data, f, indent=2, ensure_ascii=False)


def prepare_voc2012(data_root, downloads_dir, url):
    data_root = Path(data_root)
    if (data_root / "JPEGImages").exists():
        print("VOC2012 already exists:", data_root)
        return data_root

    downloads_dir = ensure_dir(downloads_dir)
    archive_path = downloads_dir / Path(url).name
    if not archive_path.exists():
        print("Downloading:", url)
        urllib.request.urlretrieve(url, archive_path)

    print("Extracting:", archive_path)
    with tarfile.open(archive_path, "r") as tar:
        tar.extractall(data_root.parent)
    print("Dataset ready:", data_root)
    return data_root


def get_device():
    return torch.device(EXPERIMENT_CONFIG["device"])


def colorize_mask(mask):
    palette = []
    for color in VOC_COLORMAP:
        palette.extend(color)
    palette.extend([0] * (768 - len(palette)))
    image = Image.fromarray(mask.astype(np.uint8), mode="P")
    image.putpalette(palette)
    return image


def blend_overlay(image, mask, alpha=0.6):
    color_mask = colorize_mask(mask).convert("RGB")
    return Image.blend(image.convert("RGB"), color_mask, alpha=alpha)


In [ ]:
class TrainTransform:
    # 训练时做随机缩放、随机裁剪和随机翻转
    def __init__(self, crop_size, min_scale, max_scale, ignore_label):
        self.crop_size = crop_size
        self.min_scale = min_scale
        self.max_scale = max_scale
        self.ignore_label = ignore_label

    def __call__(self, image, mask):
        scale = random.uniform(self.min_scale, self.max_scale)
        w = max(1, int(image.width * scale))
        h = max(1, int(image.height * scale))
        image = image.resize((w, h), Image.BILINEAR)
        mask = mask.resize((w, h), Image.NEAREST)

        pad_w = max(self.crop_size - w, 0)
        pad_h = max(self.crop_size - h, 0)
        if pad_w > 0 or pad_h > 0:
            image = ImageOps.expand(image, border=(0, 0, pad_w, pad_h), fill=0)
            mask = ImageOps.expand(mask, border=(0, 0, pad_w, pad_h), fill=self.ignore_label)

        left = random.randint(0, image.width - self.crop_size)
        top = random.randint(0, image.height - self.crop_size)
        image = image.crop((left, top, left + self.crop_size, top + self.crop_size))
        mask = mask.crop((left, top, left + self.crop_size, top + self.crop_size))

        if random.random() < 0.5:
            image = image.transpose(Image.FLIP_LEFT_RIGHT)
            mask = mask.transpose(Image.FLIP_LEFT_RIGHT)

        image = TF.to_tensor(image)
        image = TF.normalize(image, IMAGE_MEAN, IMAGE_STD)
        mask = torch.as_tensor(np.array(mask), dtype=torch.long)
        return image, mask


class EvalTransform:
    # 评估和推理时做缩放与归一化
    def __init__(self, long_size=None):
        self.long_size = long_size

    def __call__(self, image, mask=None):
        original_size = image.size
        if self.long_size is not None:
            scale = self.long_size / max(image.size)
            new_size = (int(image.width * scale), int(image.height * scale))
            image = image.resize(new_size, Image.BILINEAR)
            if mask is not None:
                mask = mask.resize(new_size, Image.NEAREST)
        image_tensor = TF.normalize(TF.to_tensor(image), IMAGE_MEAN, IMAGE_STD)
        if mask is None:
            return image_tensor, original_size
        mask_tensor = torch.as_tensor(np.array(mask), dtype=torch.long)
        return image_tensor, mask_tensor


class VOCSegmentationDataset(Dataset):
    # 读取 VOC2012 的图片、标签和文件名
    def __init__(self, data_root, split, transform):
        self.data_root = Path(data_root)
        self.transform = transform
        split_file = self.data_root / "ImageSets" / "Segmentation" / f"{split}.txt"
        self.ids = [line.strip() for line in split_file.read_text(encoding="utf-8").splitlines() if line.strip()]

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, index):
        image_id = self.ids[index]
        image = Image.open(self.data_root / "JPEGImages" / f"{image_id}.jpg").convert("RGB")
        mask = Image.open(self.data_root / "SegmentationClass" / f"{image_id}.png")
        image, mask = self.transform(image, mask)
        return image, mask, image_id


class SegmentationMetric:
    # 用混淆矩阵计算 pixel accuracy 和 mIoU
    def __init__(self, num_classes, ignore_label):
        self.num_classes = num_classes
        self.ignore_label = ignore_label
        self.confusion_matrix = np.zeros((num_classes, num_classes), dtype=np.int64)

    def update(self, prediction, target):
        prediction = prediction.cpu().numpy().astype(np.int64)
        target = target.cpu().numpy().astype(np.int64)
        valid = target != self.ignore_label
        prediction = prediction[valid]
        target = target[valid]
        hist = np.bincount(
            self.num_classes * target.reshape(-1) + prediction.reshape(-1),
            minlength=self.num_classes ** 2,
        ).reshape(self.num_classes, self.num_classes)
        self.confusion_matrix += hist

    def compute(self):
        hist = self.confusion_matrix
        pixel_accuracy = np.diag(hist).sum() / max(hist.sum(), 1)
        iou = np.diag(hist) / np.maximum(hist.sum(axis=1) + hist.sum(axis=0) - np.diag(hist), 1)
        return {
            "pixel_accuracy": float(pixel_accuracy),
            "mean_iou": float(np.nanmean(iou)),
            "class_iou": {name: float(iou[i]) for i, name in enumerate(VOC_CLASSES)},
        }


In [ ]:
def build_model():
    weights = DeepLabV3_ResNet50_Weights.COCO_WITH_VOC_LABELS_V1 if TRAIN_CONFIG["weights"] == "voc" else None
    backbone_weights = None
    if weights is None and TRAIN_CONFIG["backbone_weights"] == "imagenet":
        backbone_weights = getattr(ResNet50_Weights, "IMAGENET1K_V2", ResNet50_Weights.IMAGENET1K_V1)

    model = deeplabv3_resnet50(
        weights=weights,
        weights_backbone=backbone_weights,
        num_classes=TRAIN_CONFIG["num_classes"],
        aux_loss=True,
    )
    return model


def evaluate_model(model, loader, device):
    model.eval()
    criterion = nn.CrossEntropyLoss(ignore_index=TRAIN_CONFIG["ignore_label"])
    metric = SegmentationMetric(TRAIN_CONFIG["num_classes"], TRAIN_CONFIG["ignore_label"])
    loss_sum = 0.0
    sample_count = 0

    with torch.no_grad():
        for images, masks, _ in loader:
            images = images.to(device)
            masks = masks.to(device)
            logits = model(images)["out"]
            if logits.shape[-2:] != masks.shape[-2:]:
                logits = F.interpolate(logits, size=masks.shape[-2:], mode="bilinear", align_corners=False)
            loss = criterion(logits, masks)
            preds = logits.argmax(dim=1)
            metric.update(preds, masks)
            loss_sum += loss.item() * images.size(0)
            sample_count += images.size(0)

    metrics = metric.compute()
    metrics["loss"] = loss_sum / max(sample_count, 1)
    return metrics


def run_training(experiment_dir):
    device = get_device()
    train_dir = ensure_dir(experiment_dir / "train")
    print(f"[Train] device={device}  experiment={experiment_dir.name}")

    train_dataset = VOCSegmentationDataset(
        EXPERIMENT_CONFIG["data_root"],
        TRAIN_CONFIG["train_split"],
        TrainTransform(
            TRAIN_CONFIG["crop_size"],
            TRAIN_CONFIG["min_scale"],
            TRAIN_CONFIG["max_scale"],
            TRAIN_CONFIG["ignore_label"],
        ),
    )
    val_dataset = VOCSegmentationDataset(
        EXPERIMENT_CONFIG["data_root"],
        TRAIN_CONFIG["val_split"],
        EvalTransform(),
    )
    train_loader = DataLoader(train_dataset, batch_size=TRAIN_CONFIG["batch_size"], shuffle=True, num_workers=TRAIN_CONFIG["num_workers"])
    val_loader = DataLoader(val_dataset, batch_size=1, shuffle=False, num_workers=TRAIN_CONFIG["num_workers"])

    model = build_model().to(device)
    criterion = nn.CrossEntropyLoss(ignore_index=TRAIN_CONFIG["ignore_label"])
    optimizer = SGD(model.parameters(), lr=TRAIN_CONFIG["lr"], momentum=0.9, weight_decay=1e-4)

    history = []
    best_miou = -1.0
    for epoch in range(1, TRAIN_CONFIG["epochs"] + 1):
        model.train()
        train_loss_sum = 0.0
        train_count = 0

        for images, masks, _ in train_loader:
            images = images.to(device)
            masks = masks.to(device)
            optimizer.zero_grad()
            output = model(images)
            logits = output["out"]
            loss = criterion(logits, masks)
            if "aux" in output:
                loss = loss + 0.4 * criterion(output["aux"], masks)
            loss.backward()
            optimizer.step()
            train_loss_sum += loss.item() * images.size(0)
            train_count += images.size(0)

        train_loss = train_loss_sum / max(train_count, 1)
        val_metrics = evaluate_model(model, val_loader, device)
        history.append({
            "epoch": epoch,
            "train_loss": train_loss,
            "val_loss": val_metrics["loss"],
            "pixel_accuracy": val_metrics["pixel_accuracy"],
            "mean_iou": val_metrics["mean_iou"],
        })
        print(
            f"[Train] epoch {epoch:02d}/{TRAIN_CONFIG['epochs']:02d} | "
            f"train_loss={train_loss:.4f} | val_loss={val_metrics['loss']:.4f} | mIoU={val_metrics['mean_iou']:.4f}"
        )

        torch.save(model.state_dict(), train_dir / "last.pth")
        if val_metrics["mean_iou"] > best_miou:
            best_miou = val_metrics["mean_iou"]
            torch.save(model.state_dict(), train_dir / "best.pth")

    save_json(history, train_dir / "history.json")
    print(f"[Train] best_mIoU={best_miou:.4f}")


def run_evaluation(experiment_dir):
    device = get_device()
    evaluate_dir = ensure_dir(experiment_dir / "evaluate")
    print(f"[Eval] experiment={experiment_dir.name}")

    dataset = VOCSegmentationDataset(EXPERIMENT_CONFIG["data_root"], EVALUATE_CONFIG["split"], EvalTransform())
    loader = DataLoader(dataset, batch_size=1, shuffle=False, num_workers=TRAIN_CONFIG["num_workers"])

    model = build_model().to(device)
    model.load_state_dict(torch.load(experiment_dir / "train" / "best.pth", map_location=device))
    metrics = evaluate_model(model, loader, device)
    save_json(metrics, evaluate_dir / "metrics.json")
    print(f"[Eval] loss={metrics['loss']:.4f} | pixel_acc={metrics['pixel_accuracy']:.4f} | mIoU={metrics['mean_iou']:.4f}")
    return metrics


def run_prediction(experiment_dir):
    device = get_device()
    predict_dir = ensure_dir(experiment_dir / "predict")
    print(f"[Predict] experiment={experiment_dir.name}")

    model = build_model().to(device)
    model.load_state_dict(torch.load(experiment_dir / "train" / "best.pth", map_location=device))
    model.eval()

    input_path = Path(PREDICT_CONFIG["input"])
    if input_path.is_file():
        image_paths = [input_path]
    else:
        image_paths = sorted(input_path.glob("*.jpg"))[: PREDICT_CONFIG["max_images"]]

    transform = EvalTransform(PREDICT_CONFIG["long_size"])
    records = []
    for image_path in image_paths:
        image = Image.open(image_path).convert("RGB")
        image_tensor, original_size = transform(image)
        image_tensor = image_tensor.unsqueeze(0).to(device)

        with torch.no_grad():
            logits = model(image_tensor)["out"]
            logits = F.interpolate(logits, size=(original_size[1], original_size[0]), mode="bilinear", align_corners=False)
            prediction = logits.argmax(dim=1).squeeze(0).cpu().numpy()

        mask = colorize_mask(prediction)
        overlay = blend_overlay(image, prediction)
        mask_path = predict_dir / f"{image_path.stem}_mask.png"
        overlay_path = predict_dir / f"{image_path.stem}_overlay.png"
        mask.save(mask_path)
        overlay.save(overlay_path)
        records.append({
            "image": image_path.name,
            "mask": mask_path.name,
            "overlay": overlay_path.name,
        })

    save_json(records, predict_dir / "manifest.json")
    print(f"[Predict] saved {len(records)} images to {predict_dir}")


def run_full_experiment():
    seed_everything(EXPERIMENT_CONFIG["seed"])
    experiment_dir = build_experiment_dir()
    run_training(experiment_dir)
    metrics = run_evaluation(experiment_dir)
    run_prediction(experiment_dir)

    summary = {
        "experiment_name": experiment_dir.name,
        "metrics": metrics,
        "train_config": TRAIN_CONFIG,
    }
    save_json(summary, experiment_dir / "summary.json")
    return experiment_dir, metrics


In [ ]:
# 如果本地已经存在 VOC2012 数据集，可以跳过本单元
# prepare_voc2012(**DOWNLOAD_CONFIG)

In [ ]:
# 执行完整实验：按顺序完成训练、评估和推理
experiment_dir, metrics = run_full_experiment()
print("experiment_dir:", experiment_dir)
print("mean_iou:", round(metrics["mean_iou"], 4))


In [ ]:
# 查看本次实验输出文件
for path in sorted(experiment_dir.rglob("*")):
    rel_path = path.relative_to(experiment_dir)
    prefix = "[DIR]" if path.is_dir() else "[FILE]"
    print(prefix, rel_path)


In [ ]:
# 显示一张推理叠加图
overlay_images = sorted((experiment_dir / "predict").glob("*_overlay.png"))
if overlay_images:
    image = Image.open(overlay_images[0]).convert("RGB")
    plt.figure(figsize=(8, 6))
    plt.imshow(image)
    plt.title(overlay_images[0].name)
    plt.axis("off")
    plt.show()
else:
    print("No overlay image found.")
